# 💳 Credit Scoring Model
### CodeAlpha Machine Learning Internship Project
**Intern:** Subhasmita Panda  
**Objective:** Predict creditworthiness using past financial data  
**Algorithms:** Logistic Regression, Decision Tree, Random Forest

## Step 1: Install & Import Libraries

In [ ]:
# Install if not already installed
# !pip install pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

print('✅ All libraries imported successfully!')

## Step 2: Load Dataset
> We use the **German Credit Dataset** from UCI — no download needed, loaded directly from URL.

In [ ]:
# Load German Credit Dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'

columns = [
    'checking_account', 'duration', 'credit_history', 'purpose', 'credit_amount',
    'savings_account', 'employment', 'installment_rate', 'personal_status',
    'other_debtors', 'residence_since', 'property', 'age', 'other_installments',
    'housing', 'existing_credits', 'job', 'dependents', 'telephone', 'foreign_worker',
    'creditworthiness'
]

df = pd.read_csv(url, sep=' ', header=None, names=columns)

# Convert target: 1 = Good Credit, 2 = Bad Credit → remap to 0 and 1
df['creditworthiness'] = df['creditworthiness'].map({1: 0, 2: 1})  # 0=Good, 1=Bad

print(f'✅ Dataset loaded! Shape: {df.shape}')
df.head()

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Basic Info
print('Shape:', df.shape)
print('\nMissing Values:\n', df.isnull().sum())
print('\nTarget Distribution:\n', df['creditworthiness'].value_counts())

In [ ]:
# Target Distribution Plot
plt.figure(figsize=(6, 4))
sns.countplot(x='creditworthiness', data=df, palette=['#2ecc71', '#e74c3c'])
plt.title('Credit Worthiness Distribution\n(0 = Good, 1 = Bad)', fontsize=13)
plt.xlabel('Creditworthiness')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Age distribution by credit
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x='age', hue='creditworthiness', kde=True, palette=['#2ecc71', '#e74c3c'])
plt.title('Age Distribution by Creditworthiness')
plt.tight_layout()
plt.show()

In [ ]:
# Credit Amount vs Duration
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='duration', y='credit_amount', hue='creditworthiness',
                palette=['#2ecc71', '#e74c3c'], alpha=0.6)
plt.title('Credit Amount vs Loan Duration')
plt.tight_layout()
plt.show()

## Step 4: Feature Engineering & Preprocessing

In [ ]:
# Encode all categorical columns using LabelEncoder
le = LabelEncoder()
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# Feature Engineering: Create new features
df['credit_per_duration'] = df['credit_amount'] / (df['duration'] + 1)   # Credit per month
df['age_employment_ratio'] = df['age'] / (df['employment'].replace(0, 1)) # Age vs Employment

print('✅ Encoding and feature engineering done!')
df.head()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## Step 5: Split Data & Scale Features

In [ ]:
# Define features and target
X = df.drop('creditworthiness', axis=1)
y = df['creditworthiness']

# Train-Test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'✅ Train size: {X_train.shape}, Test size: {X_test.shape}')

## Step 6: Train Models

In [ ]:
# ---- Model 1: Logistic Regression ----
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]
print('✅ Logistic Regression trained!')

# ---- Model 2: Decision Tree ----
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)
dt_proba = dt_model.predict_proba(X_test)[:, 1]
print('✅ Decision Tree trained!')

# ---- Model 3: Random Forest ----
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]
print('✅ Random Forest trained!')

## Step 7: Evaluate Models

In [ ]:
def evaluate_model(name, y_true, y_pred, y_proba):
    print(f'\n{'='*50}')
    print(f'📊 {name}')
    print(f'{'='*50}')
    print(classification_report(y_true, y_pred, target_names=['Good Credit', 'Bad Credit']))
    roc = roc_auc_score(y_true, y_proba)
    print(f'ROC-AUC Score: {roc:.4f}')
    return roc

lr_roc  = evaluate_model('Logistic Regression', y_test, lr_preds, lr_proba)
dt_roc  = evaluate_model('Decision Tree',       y_test, dt_preds, dt_proba)
rf_roc  = evaluate_model('Random Forest',       y_test, rf_preds, rf_proba)

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
models = [
    ('Logistic Regression', lr_preds),
    ('Decision Tree', dt_preds),
    ('Random Forest', rf_preds)
]

for ax, (name, preds) in zip(axes, models):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Good', 'Bad'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=12)

plt.suptitle('Confusion Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(8, 6))

for name, proba, color in [
    ('Logistic Regression', lr_proba, '#3498db'),
    ('Decision Tree',       dt_proba, '#e67e22'),
    ('Random Forest',       rf_proba, '#2ecc71')
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', color=color, lw=2)

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8: Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(9, 5))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Top 15 Feature Importances — Random Forest', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## Step 9: Final Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'ROC-AUC': [lr_roc, dt_roc, rf_roc]
}).sort_values('ROC-AUC', ascending=False)

print('\n🏆 MODEL COMPARISON')
print(results.to_string(index=False))

best = results.iloc[0]['Model']
print(f'\n✅ Best Model: {best} with ROC-AUC = {results.iloc[0]["ROC-AUC"]:.4f}')

In [ ]:
# Bar chart comparison
plt.figure(figsize=(7, 4))
sns.barplot(data=results, x='Model', y='ROC-AUC', palette=['#2ecc71', '#3498db', '#e74c3c'])
plt.ylim(0.5, 1.0)
plt.title('Model ROC-AUC Comparison', fontsize=13)
plt.ylabel('ROC-AUC Score')
plt.tight_layout()
plt.show()

## ✅ Project Complete!

**Summary:**
- Loaded and explored the German Credit Dataset (1000 records, 20+ features)
- Performed feature engineering (credit_per_duration, age_employment_ratio)
- Trained 3 classifiers: Logistic Regression, Decision Tree, Random Forest
- Evaluated using Precision, Recall, F1-Score, ROC-AUC
- Random Forest typically achieves the best ROC-AUC score

**Intern:** Subhasmita Panda | **Organization:** CodeAlpha